In [1]:
import pandas as pd
import plotly.express as px
import numpy as np

# Dataset: Global Energy Mix by Country and Source
df = pd.read_csv('global_energy_mix.csv')

# Source type mapping — reuse from lecture
source_category = {
    'Coal': 'Fossil', 'Oil': 'Fossil', 'Natural Gas': 'Fossil',
    'Nuclear': 'Low-carbon', 'Hydro': 'Low-carbon',
    'Wind': 'Renewable', 'Solar': 'Renewable', 'Other Renewables': 'Renewable'
}
df['Source_Type'] = df['Source'].map(source_category)

print(f"Loaded: {len(df)} rows")
print(df.head(10))

Loaded: 103 rows
         Country         Region            Source  Share_pct     TWh  \
0  United States  North America              Coal         10  1015.0   
1  United States  North America               Oil         35  3220.0   
2  United States  North America       Natural Gas         34  3083.0   
3  United States  North America           Nuclear          9   798.0   
4  United States  North America             Hydro          3   339.0   
5  United States  North America              Wind          4   413.0   
6  United States  North America             Solar          3   325.0   
7  United States  North America  Other Renewables          2   229.0   
8          China           Asia              Coal         60  7168.0   
9          China           Asia               Oil         18  1620.0   

  Source_Type  
0      Fossil  
1      Fossil  
2      Fossil  
3  Low-carbon  
4  Low-carbon  
5   Renewable  
6   Renewable  
7   Renewable  
8      Fossil  
9      Fossil  


In [10]:





fossil = df.loc[df['Source_Type'] == 'Fossil'].copy()

print(f"Fossil rows: {len(fossil)}")
print(fossil['Source'].unique())
print(fossil.groupby('Region')['TWh'].sum().sort_values(ascending=False))


# Step 2: Define CVD-safe colour palette for the three fossil sources
# Using a palette safe for deuteranopia/protanopia (the most common forms):
#   Coal        → dark charcoal/brown  (distinguishable by lightness)
#   Oil         → orange               (CVD-safe warm tone)
#   Natural Gas → sky blue             (CVD-safe cool tone, distinct from orange)

fossil_color_map = {
    'Coal':        '#4E3B2E',   # dark brown/charcoal — heaviest emitter
    'Oil':         '#E07B39',   # orange — CVD-safe warm
    'Natural Gas': '#56B4E9',   # sky blue — CVD-safe cool (from Wong palette)
}

# ── Step 3: Plotly Express base chart ─────────────────────────────────────────
fig = px.treemap(
    fossil,
    path=['Region', 'Country', 'Source'],   # Region → Country → Fossil Source
    values='TWh',
    color='Source',                          # colour encodes fossil source type
    color_discrete_map=fossil_color_map,
    labels={'TWh': 'Energy (TWh)', 'Source': 'Fossil Source'},
    height=750, width=1300
)

# ── Step 4: Customise traces ──────────────────────────────────────────────────
fig.update_traces(
    textinfo='label+value',
    # Show TWh values — NO percentages, thousands separator, no decimals
    texttemplate='%{label}<br>%{value:,.0f} TWh',
    hovertemplate='<b>%{label}</b><br>%{value:,.0f} TWh<extra></extra>',
)

# Grey out parent nodes (Region and Country level)
# Colours list has entries for all nodes; leaf nodes get the palette colour,
# parent nodes (Region / Country) fall back to the discrete map as '#ffffff' or
# whatever Plotly assigned — we override non-leaf colours to grey.
leaf_colors = set(fossil_color_map.values())
fig.data[0].marker.colors = [
    c if c in leaf_colors else '#DDDDDD'
    for c in fig.data[0].marker.colors
]

# ── Step 5: Layout ────────────────────────────────────────────────────────────
fig.update_layout(
    title=dict(
        text=(
            'Asia Pacific burns the most fossil fuels — '
            'coal dominates over oil and gas across the region'
        ),
        font=dict(family='Arial', size=15, color='#222222')
    ),
    font=dict(family='Arial', size=12),
    margin=dict(l=10, r=10, t=60, b=10),
    paper_bgcolor='white',
)

fig.show()

Fossil rows: 43
['Coal' 'Oil' 'Natural Gas']
Region
Asia             35284.0
Europe           15387.0
Latin America    12376.0
North America    11131.0
Middle East       9671.0
Africa            8620.0
Oceania           8142.0
Name: TWh, dtype: float64


In [6]:
import pandas as pd
import plotly.express as px



# Load dataset
df = pd.read_csv('global_energy_mix.csv')

# Source type mapping
source_category = {
    'Coal': 'Fossil',
    'Oil': 'Fossil',
    'Natural Gas': 'Fossil',
    'Nuclear': 'Low-carbon',
    'Hydro': 'Low-carbon',
    'Wind': 'Renewable',
    'Solar': 'Renewable',
    'Other Renewables': 'Renewable'
}

df['Source_Type'] = df['Source'].map(source_category)

# ---------------------------------------------------
# Filter only Low-carbon sources
# ---------------------------------------------------

low_carbon_df = df[df['Source_Type'] == 'Low-carbon']

# Aggregate TWh by country
country_lowcarbon = (
    low_carbon_df.groupby('Country')['TWh']
    .sum()
    .reset_index()
    .sort_values(by='TWh', ascending=False)
)

# Add dummy root node for treemap
country_lowcarbon['All'] = 'Low-carbon'

# Leading country
top_country = country_lowcarbon.iloc[0]['Country']

# ---------------------------------------------------
# LIGHT COLOUR PALETTE
# ---------------------------------------------------

light_palette = [
    '#A8DADC',
    '#BDE0FE',
    '#CDB4DB',
    '#FFD6A5',
    '#CAFFBF',
    '#FFC8DD',
    '#D8E2DC',
    '#E9EDC9'
]

# ---------------------------------------------------
# TREEMAP
# ---------------------------------------------------

treemap_fig = px.treemap(
    country_lowcarbon,
    path=['All', 'Country'],
    values='TWh',
    color='TWh',
    color_continuous_scale='Blues',
    title='Low-carbon Energy Production by Country (Treemap)'
)

# Treemap styling
treemap_fig.update_traces(
    textinfo='label+value',
    root_color='lightgrey',
    hovertemplate=
    '<b>%{label}</b><br>' +
    'Low-carbon Energy: %{value:,.0f} TWh<extra></extra>'
)

treemap_fig.update_layout(
    width=1000,
    height=700,

    margin=dict(t=70, l=20, r=20, b=20),

    title_font=dict(size=22),

    font=dict(
        family='Arial',
        size=14
    )
)

treemap_fig.show()

# ---------------------------------------------------
# HORIZONTAL BAR CHART
# ---------------------------------------------------

bar_fig = px.bar(
    country_lowcarbon,
    x='TWh',
    y='Country',
    orientation='h',

    color='TWh',
    color_continuous_scale='Tealgrn',

    text='TWh',

    title=(
        f'Low-carbon Energy Production by Country'
        f'<br><sup>{top_country} is the leading producer '
        f'of low-carbon electricity.</sup>'
    )
)

# Sort highest at top
bar_fig.update_layout(
    yaxis=dict(
        categoryorder='total ascending'
    )
)

# Bar styling
bar_fig.update_traces(
    texttemplate='%{text:,.0f} TWh',
    textposition='outside',

    hovertemplate=
    '<b>%{y}</b><br>' +
    'Low-carbon Energy: %{x:,.0f} TWh<extra></extra>'
)

bar_fig.update_layout(
    width=1100,
    height=800,

    margin=dict(t=90, l=120, r=40, b=40),

    title_font=dict(size=24),

    font=dict(
        family='Arial',
        size=14
    ),

    xaxis_title='Low-carbon Energy (TWh)',
    yaxis_title='Country',

    plot_bgcolor='white'
)

bar_fig.show()

# ---------------------------------------------------
# SUMMARY TABLE
# ---------------------------------------------------

print("Top 10 Low-carbon Energy Producers")
print(country_lowcarbon.head(10))

Top 10 Low-carbon Energy Producers
           Country     TWh         All
4           France  7593.0  Low-carbon
9           Norway  6174.0  Low-carbon
2           Canada  5969.0  Low-carbon
1           Brazil  4975.0  Low-carbon
11     South Korea  2484.0  Low-carbon
6            India  1336.0  Low-carbon
13   United States  1137.0  Low-carbon
7            Japan  1012.0  Low-carbon
12  United Kingdom   993.0  Low-carbon
5          Germany   929.0  Low-carbon
